# Proyecto: Comunidad energética

## Descripción del proyecto

**Objetivo**: consiste en repartir la energía entre los participantes de una comunidad energética.
La energía se puede repartir con uno dos o tres criterios no excluyentes: dinero invertido, superficie disponible, gasto energético. El peso de los criterios se denomina PONDERACIÓN (son porcentajes).
Se trabajará con un conjunto de ficheros cuyo nombre será un identificador llamado CUPS. Cada fichero lista por hora el consumo energético del participante. Por otro lado, se trabajará con un único fichero en el que se listan el dinero aportado por cada participante (identificado con su CUPS) y otro único fichero en el que se lista la superficie de cada participante-CUPS.

**Resultado**: un CSV. Columnas: CUPS, coeficiente beta
El coeficiente beta es el tanto por ciento de energía que le toca a cada participante cada hora del año. Cómo se calcula: es el resultado de tres sumandos.
Sumando 1: (Ponderación del criterio 1) por (consumo de ese vecino a esa hora) dividido entre (consumo total de todos los vecinos)
Sumando 2: (Ponderación del criterio 2) por (dinero aportado) dividido entre (dinero total aportado por la comunidad)
Sumando 3: (Ponderación del criterio 3) por (superficie aportada) dividido entre (superficie total de la comunidad)


CRITERIO 1: consumo eléctrico de cada participante
El consumo de cada uno no es expresa como el total anual que consume, sino por hora de cada día del año.
El total de la energía tiene que sumar 1, y durante los cálculos se tiene que operar con 6 decimales.

CRITERIO 2: dinero
Cada participante pone una suma de dinero distinta. Obviamente, habrá tantas sumas de dinero aportadas como participantes p.

CRITERIO 3: superficie (m2)


VARIABLES:
p: el número total de participantes del proyecto.
Numero de horas en un año: 8760

## Generación de datos

In [11]:
import numpy as np

In [12]:
vecino_1_CUPS = 'ES01AA1A'
vecino_2_CUPS = 'ES01AA1B'

horas = 8760

consumos_1 = np.round(np.random.uniform(0.0, 400.0, size=horas), 4)
consumos_2 = np.round(np.random.uniform(0.0, 400.0, size=horas), 4)
consumos_totales = np.sum(consumos_1 + consumos_2)

dinero_1 = 250
dinero_2 = 150
dinero_total = dinero_1 + dinero_2

superficie_1 = 50
superficie_2 = 60
superficie_total = superficie_1 + superficie_2

In [14]:
print('consumos 1:', consumos_1)
print('consumo total vecino 1:', np.sum(consumos_1))
print('consumos 2:', consumos_2)
print('consumo total vecino 2:', np.sum(consumos_2))
print('consumos totales:', consumos_totales)

consumos 1: [311.7779  83.7157  95.2485 ... 296.821   82.1963 393.0733]
consumo total vecino 1: 1740983.0583
consumos 2: [292.4086 181.4104  47.9353 ... 152.7236 333.9271 261.6118]
consumo total vecino 2: 1752506.8297
consumos totales: 3493489.888


In [15]:
# Ponderaciones
crit_1 = 0.5
crit_2 = 0.25
crit_3 = 0.25

## Cálculos

In [16]:
betas_1 = consumos_1 * crit_1 / consumos_totales + crit_2 * dinero_1 / dinero_total / horas + crit_3 * superficie_1 / superficie_total / horas
betas_2 = consumos_2 * crit_1 / consumos_totales + crit_2 * dinero_2 / dinero_total / horas + crit_3 * superficie_2 / superficie_total / horas
print(betas_1)
print(betas_2)

[7.54316453e-05 4.27906176e-05 4.44412307e-05 ... 7.32909635e-05
 4.25731560e-05 8.70669158e-05]
[6.81191804e-05 5.22327454e-05 3.31293410e-05 ... 4.81269943e-05
 7.40614475e-05 6.37114390e-05]


## Generación de tabla

In [18]:
import pandas as pd

In [19]:
betas_1_df = pd.DataFrame(data={'CUPS':np.array([vecino_1_CUPS for i in range(horas)]), 'betas':betas_1})
betas_2_df = pd.DataFrame(data={'CUPS':np.array([vecino_2_CUPS for i in range(horas)]), 'betas':betas_2})

In [20]:
betas_vecinos = pd.concat([betas_1_df, betas_2_df], ignore_index=True)
betas_vecinos

,CUPS,betas
0,ES01AA1A,0.000075
1,ES01AA1A,0.000043
2,ES01AA1A,0.000044
3,ES01AA1A,0.000074
4,ES01AA1A,0.000037
...,...,...
17515,ES01AA1B,0.000037
17516,ES01AA1B,0.000075
17517,ES01AA1B,0.000048
17518,ES01AA1B,0.000074


In [21]:
betas_vecinos['betas'].sum()

np.float64(1.0)

## Lectura de documentos

In [2]:
import pandas as pd
import os

In [118]:
# generar dataframe vacío
df_final = pd.DataFrame(columns=['CUPS', 'betas'])

In [3]:
# leer directorio consumos
cups_files = os.listdir(os.getcwd() + '/casos/consumos')
cups_names = [i[:-4] for i in cups_files]

In [112]:
# obtener total de consumos
consumos_totales = 0

for i in cups_files:
    df = pd.read_csv(os.getcwd() + '/casos/consumos/' + i, sep=';')
    df['Consumo'] = df['Consumo'].str.replace(',', '.').astype(float)
    consumos_totales += df['Consumo'].sum()

consumos_totales

np.float64(244312.11400000012)

In [113]:
# leer archivo dinero y sacar aportacion
df_aportacion = pd.read_csv(os.getcwd() + '/casos/aportaciones.csv', header=None, names=['CUPS', 'aportacion'])
aportacion_total = df_aportacion['aportacion'].sum()
vecino_aportacion = df_aportacion[df_aportacion['CUPS'] == cups_names[1]]['aportacion'].iloc[0]

In [114]:
# leer archivo superficie y sacar superficie
df_superficie = pd.read_csv(os.getcwd() + '/casos/superficies.csv', header=None, names=['CUPS', 'superficie'])
superficie_total = df_superficie['superficie'].sum()
vecino_superficie = df_superficie[df_superficie['CUPS'] == cups_names[1]]['superficie'].iloc[0]

In [115]:
# leer archivo consumos y sacar consumo
vecino_consumo = pd.read_csv(os.getcwd() + '/casos/consumos/' + cups_files[1], sep=';')
vecino_consumo['Consumo'] = vecino_consumo['Consumo'].str.replace(',', '.').astype(float)

In [ ]:
# hacer cálculos
# consumos_1 * crit_1 / consumos_totales + crit_2 * dinero_1 / dinero_total / horas + crit_3 * superficie_1 / superficie_total / horas
sumando_1 = vecino_consumo['Consumo'].iloc[-8760:] * crit_1 / consumos_totales
sumando_2 = vecino_aportacion * crit_2 / aportacion_total / 8760
sumando_3 = vecino_superficie * crit_3 / superficie_total / 8760
vecino_beta = sumando_1 + sumando_2 + sumando_3
vecino_beta

In [23]:
import os
import numpy as np
import pandas as pd

# generar dataframe vacío
df_final = pd.DataFrame(columns=['CUPS', 'betas'])

# leer directorio consumos
cups_files = os.listdir(os.getcwd() + '/casos/consumos')
cups_names = [i[:-4] for i in cups_files]

# obtener total de consumos
consumos_totales = 0
for i in cups_files:
    df = pd.read_csv(os.getcwd() + '/casos/consumos/' + i, sep=';')
    df['Consumo'] = df['Consumo'].str.replace(',', '.').astype(float)
    consumos_totales += df['Consumo'].sum()

# leer archivo dinero y sacar aportacion total
df_aportacion = pd.read_csv(os.getcwd() + '/casos/aportaciones.csv', header=None, names=['CUPS', 'aportacion'])
aportacion_total = df_aportacion['aportacion'].sum()

# leer archivo superficie y sacar superficie total
df_superficie = pd.read_csv(os.getcwd() + '/casos/superficies.csv', header=None, names=['CUPS', 'superficie'])
superficie_total = df_superficie['superficie'].sum()

# búsqueda de cups en aportaciones y superficies, lectura y procesado de consumo, calculo de betas y concatenado
for i in cups_names:
    # variables
    vecino_aportacion = df_aportacion[df_aportacion['CUPS'] == i]['aportacion'].iloc[0]
    vecino_superficie = df_superficie[df_superficie['CUPS'] == i]['superficie'].iloc[0]
    vecino_consumo = pd.read_csv(os.getcwd() + '/casos/consumos/' + i + '.csv', sep=';')
    vecino_consumo['Consumo'] = vecino_consumo['Consumo'].str.replace(',', '.').astype(float)
    if len(vecino_consumo) < 8760:
       print(i)
       vecino_consumo = pd.concat([vecino_consumo,
                                   pd.DataFrame({'Consumo':[0 for e in range(8760-len(vecino_consumo))]})])
    # betas
    sumando_1 = vecino_consumo['Consumo'].iloc[-8760:] * crit_1 / consumos_totales
    sumando_2 = vecino_aportacion * crit_2 / aportacion_total / 8760
    sumando_3 = vecino_superficie * crit_3 / superficie_total / 8760
    vecino_beta = sumando_1 + sumando_2 + sumando_3
    # concatenar
    df_final = pd.concat([df_final,
                          pd.DataFrame({'CUPS':[i for j in range(8760)],
                                        'betas': vecino_beta})],
                         ignore_index=True)

print(df_final['betas'].sum())
df_final

/var/folders/c1/57t2nrh50392nthwkbjd_8sm0000gn/T/ipykernel_1212/2476425673.py:44: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat([df_final,


ES0022000008485886QT1P
0.9620168138694914


,CUPS,betas
0,ES0000000000006457CCPF,6.473282e-07
1,ES0000000000006457CCPF,6.104901e-07
2,ES0000000000006457CCPF,6.145832e-07
3,ES0000000000006457CCPF,6.411885e-07
4,ES0000000000006457CCPF,6.350488e-07
...,...,...
1033675,ES0000000000003582CCPF,7.961114e-07
1033676,ES0000000000003582CCPF,8.963930e-07
1033677,ES0000000000003582CCPF,1.643388e-06
1033678,ES0000000000003582CCPF,1.590178e-06


In [ ]:
'''PRUEBA CON SÓLO 2 .CSV
csv con menos de 8760
'''

import os
import numpy as np
import pandas as pd

# generar dataframe vacío
df_final = pd.DataFrame(columns=['CUPS', 'betas'])

# leer directorio consumos
# cups_files = os.listdir(os.getcwd() + '/casos/consumos')
cups_names = ['ES0000000000006457CCPF', 'ES0000000000006516CCPF']

# obtener total de consumos
consumos_totales = 0
for i in cups_names:
    df = pd.read_csv(os.getcwd() + '/casos/consumos/' + i + '.csv', sep=';')
    df['Consumo'] = df['Consumo'].str.replace(',', '.').astype(float)
    consumos_totales += df['Consumo'].sum()

# leer archivo dinero y sacar aportacion total
df_aportacion = pd.read_csv(os.getcwd() + '/casos/aportaciones.csv', header=None, names=['CUPS', 'aportacion'])
aportacion_total = df_aportacion[df_aportacion['CUPS'] == 'ES0000000000006457CCPF']['aportacion'].iloc[0] + df_aportacion[df_aportacion['CUPS'] == 'ES0022000008485886QT1P']['aportacion'].iloc[0]

# leer archivo superficie y sacar superficie total
df_superficie = pd.read_csv(os.getcwd() + '/casos/superficies.csv', header=None, names=['CUPS', 'superficie'])
superficie_total = df_superficie[df_superficie['CUPS'] == 'ES0000000000006457CCPF']['superficie'].iloc[0] + df_superficie[df_superficie['CUPS'] == 'ES0022000008485886QT1P']['superficie'].iloc[0]

# búsqueda de cups en aportaciones y superficies, lectura y procesado de consumo, calculo de betas y concatenado
for i in cups_names:
    # variables
    vecino_aportacion = df_aportacion[df_aportacion['CUPS'] == i]['aportacion'].iloc[0]
    vecino_superficie = df_superficie[df_superficie['CUPS'] == i]['superficie'].iloc[0]
    vecino_consumo = pd.read_csv(os.getcwd() + '/casos/consumos/' + i + '.csv', sep=';')
    vecino_consumo['Consumo'] = vecino_consumo['Consumo'].str.replace(',', '.').astype(float)
    if len(vecino_consumo) < 8760:
       print(i)
       vecino_consumo = pd.concat([vecino_consumo,
                                   pd.DataFrame({'Consumo':[0 for e in range(8760-len(vecino_consumo))]})])
    # betas
    sumando_1 = vecino_consumo['Consumo'].iloc[-8760:] * crit_1 / consumos_totales
    sumando_2 = vecino_aportacion * crit_2 / aportacion_total / 8760
    sumando_3 = vecino_superficie * crit_3 / superficie_total / 8760
    vecino_beta = sumando_1 + sumando_2 + sumando_3
    # concatenar
    df_final = pd.concat([df_final,
                          pd.DataFrame({'CUPS':[i for j in range(8760)],
                                        'betas': vecino_beta})],
                         ignore_index=True)

print(df_final.betas.sum())
df_final

0.9160884589300786


/var/folders/c1/57t2nrh50392nthwkbjd_8sm0000gn/T/ipykernel_1212/2683351213.py:47: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat([df_final,


,CUPS,betas
0,ES0000000000006457CCPF,0.000037
1,ES0000000000006457CCPF,0.000035
2,ES0000000000006457CCPF,0.000035
3,ES0000000000006457CCPF,0.000037
4,ES0000000000006457CCPF,0.000036
...,...,...
17515,ES0000000000006516CCPF,0.000034
17516,ES0000000000006516CCPF,0.000039
17517,ES0000000000006516CCPF,0.000085
17518,ES0000000000006516CCPF,0.000065


In [9]:
df_final.betas

0        0.000037
1        0.000035
2        0.000035
3        0.000037
4        0.000036
           ...   
17515    0.000034
17516    0.000039
17517    0.000085
17518    0.000065
17519    0.000035
Name: betas, Length: 17520, dtype: float64

In [6]:
df_final.betas.sum()

np.float64(0.9160884589300786)